# SFT Multi-Source Sharded DataLoader — Pipeline Test v2

| Step | What happens |
|------|--------------|
| 1 | Clone repo + install LibTorch + Python deps |
| 2 | Stream 3 datasets directly to JSONL (zero RAM buffering) |
| 3 | Tokenize 3 JSONL sources → shards |
| 4 | Verify shards and manifest |
| 5 | Build C++ binary |
| 6 | Run binary — loads shards, builds batches, trims padding |

**3 datasets:**
- `tatsu-lab/alpaca` — instruction+input → prompt, output → response
- `HuggingFaceH4/ultrachat_200k` — messages[user] → prompt, messages[assistant] → response
- `CohereLabs/aya_dataset` — inputs → prompt, targets → response

## 0. Config

In [2]:
import os

REPO_URL      = "https://github.com/SniAssia/Optimized_data_loaders.git"
REPO_DIR      = "Optimized_data_loaders"
SFT_DIR       = os.path.join(REPO_DIR, "sft_multiple_datasets")
LIBTORCH_PATH = "/content/libtorch"

TOKENIZER   = "inceptionai/jais-family-590m"
MAX_SEQ_LEN = 512
SHARD_SIZE  = 5_000
SHUFFLE_BUF = 20_000
MAX_RECORDS = None   # records per source — set to None for full dataset

print(f"SFT_DIR     : {SFT_DIR}")
print(f"MAX_SEQ_LEN : {MAX_SEQ_LEN}")
print(f"SHARD_SIZE  : {SHARD_SIZE:,}")
print(f"MAX_RECORDS : {MAX_RECORDS if MAX_RECORDS else 'unlimited (full dataset)'} per source")

SFT_DIR     : Optimized_data_loaders/sft_multiple_datasets
MAX_SEQ_LEN : 512
SHARD_SIZE  : 5,000
MAX_RECORDS : unlimited (full dataset) per source


## 1. Clone / Pull Repo

In [3]:
import subprocess

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

print("\nsft_multiple_datasets contents:")
for f in sorted(os.listdir(SFT_DIR)):
    print(" ", f)


sft_multiple_datasets contents:
  CMakeLists.txt
  collator.h
  dataloader.h
  dataset.h
  distributed.h
  main.cpp
  prefetcher.h
  readme.md
  requirements.txt
  tokenize_dataset.py


## 2. Verify Required Files

In [5]:
required = [
    "tokenize_dataset.py",
    "dataset.h", "collator.h",
    "prefetcher.h", "dataloader.h", "distributed.h",
    "main.cpp", "CMakeLists.txt"
]

all_ok = True
for fname in required:
    path = os.path.join(SFT_DIR, fname)
    ok   = os.path.isfile(path)
    print(f"  {'OK' if ok else 'MISSING':8} {fname}")
    if not ok:
        all_ok = False

assert all_ok, "Some required files are missing — check the repo."
print("\nAll required files present.")

  OK       tokenize_dataset.py
  OK       dataset.h
  OK       collator.h
  OK       prefetcher.h
  OK       dataloader.h
  OK       distributed.h
  OK       main.cpp
  OK       CMakeLists.txt

All required files present.


## 3. Install Python Dependencies

In [6]:
subprocess.run(
    ["pip", "install", "-q", "datasets", "transformers"],
    check=True
)
print("datasets + transformers: installed")

datasets + transformers: installed


## 4. LibTorch

In [7]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA:   ", torch.version.cuda)

if not os.path.isdir(LIBTORCH_PATH):
    !wget -q https://download.pytorch.org/libtorch/nightly/cu128/libtorch-cxx11-abi-shared-with-deps-latest.zip
    !unzip -q libtorch-cxx11-abi-shared-with-deps-latest.zip
    print("libtorch ready at", LIBTORCH_PATH)
else:
    print("libtorch already present")

PyTorch: 2.11.0+cu128
CUDA:    12.8
libtorch ready at /content/libtorch


In [8]:
with open(os.path.join(SFT_DIR, "collator.h"), "r") as f:
    content = f.read()

# fix missing < after std::tuple in operator() signature
fixed = content.replace(
    "const std::vector<std::tuple\n",
    "const std::vector<std::tuple<\n"
)

assert fixed != content, "Pattern not found — print collator.h to check exact text"

with open(os.path.join(SFT_DIR, "collator.h"), "w") as f:
    f.write(fixed)

print("fixed — rerun the build cell")

fixed — rerun the build cell


In [10]:
with open(os.path.join(SFT_DIR, "prefetcher.h"), "r") as f:
    content = f.read()

# count occurrences to fix
before = content.count("using CollatorItem = std::tuple\n")
print(f"found {before} occurrence(s) to fix")

fixed = content.replace(
    "using CollatorItem = std::tuple\n",
    "using CollatorItem = std::tuple<\n"
)

assert fixed != content, "Pattern not found — print prefetcher.h to check"

with open(os.path.join(SFT_DIR, "prefetcher.h"), "w") as f:
    f.write(fixed)

print("fixed — rerun build cell")

found 1 occurrence(s) to fix
fixed — rerun build cell


## 5. Build C++ Binary

In [11]:
import shutil

if not shutil.which("cmake"):
    subprocess.run(["apt-get", "install", "-y", "-q", "cmake"], check=True)

build_dir = os.path.join(SFT_DIR, "build")
os.makedirs(build_dir, exist_ok=True)

cfg_run = subprocess.run(
    ["cmake", f"-DCMAKE_PREFIX_PATH={LIBTORCH_PATH}", ".."],
    cwd=build_dir, capture_output=True, text=True,
)
print("=== cmake configure ===")
print(cfg_run.stdout[-2000:])
if cfg_run.stderr:
    print(cfg_run.stderr[-500:])
assert cfg_run.returncode == 0, "cmake configure failed"

bld = subprocess.run(
    ["cmake", "--build", ".", "-j"],
    cwd=build_dir, capture_output=True, text=True,
)
print("=== cmake build ===")
print(bld.stdout[-2000:])
if bld.stderr:
    print(bld.stderr[-1000:])
assert bld.returncode == 0, "cmake build failed"

binary_path = None
for name in ["dataloader_demo", "dataloader_test"]:
    candidate = os.path.join(build_dir, name)
    if os.path.isfile(candidate):
        binary_path = candidate
        break

assert binary_path is not None, "Binary not found after build"
print(f"\nbuild_ok = True")
print(f"binary   = {binary_path}")

=== cmake configure ===
-- PyTorch: CUDA detected: 12.8
-- PyTorch: CUDA nvcc is: /usr/local/cuda/bin/nvcc
-- PyTorch: CUDA toolkit directory: /usr/local/cuda
-- PyTorch: Header version is: 12.8
-- Could NOT find nvtx3 (missing: nvtx3_dir) 
-- USE_CUDNN is set to 0. Compiling without cuDNN support
-- USE_CUSPARSELT is set to 0. Compiling without cuSPARSELt support
-- USE_CUDSS is set to 0. Compiling without cuDSS support
-- USE_CUFILE is set to 0. Compiling without cuFile support
-- Autodetected CUDA architecture(s):  7.5
-- Added CUDA NVCC flags for: -gencode;arch=compute_75,code=sm_75
-- Configuring done (2.1s)
-- Generating done (0.0s)
-- Build files have been written to: /content/Optimized_data_loaders/sft_multiple_datasets/build

make/Torch/TorchConfig.cmake:68 (find_package)
  CMakeLists.txt:9 (find_package)
This warning is for project developers.  Use -Wno-dev to suppress it.

CMake Warning at /content/libtorch/share/cmake/Caffe2/public/cuda.cmake:184 (message):
  Cannot find NV

---
## 6. Dataset Streaming Adapters

Each adapter is a **Python generator** — it streams one record at a time
from HuggingFace and yields normalized `{prompt, response}` dicts.
No list accumulation in RAM — at any moment only one record exists in memory.

`stream_to_jsonl()` writes each record to disk immediately as it arrives
from the generator, then discards it. Peak RAM = one record (~1 KB).

In [12]:
import json
from datasets import load_dataset


def stream_to_jsonl(generator, path, max_records=None):
    """
    Streams records from generator directly to disk one at a time.
    Never accumulates records in RAM.
    Returns total records written.
    """
    n = 0
    with open(path, "w", encoding="utf-8") as f:
        for rec in generator:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
            n += 1
            if max_records and n >= max_records:
                break
            if n % 1000 == 0:
                print(f"  {n:,} records written...", end="\r", flush=True)
    print(f"  {n:,} records written.          ")
    return n


# ── Adapter 1: tatsu-lab/alpaca ──────────────────────────────
def adapt_alpaca_stream(split="train"):
    """
    columns: instruction, input, output
    prompt   = instruction + newline + input  (if input non-empty)
    response = output
    """
    ds = load_dataset("tatsu-lab/alpaca", split=split, streaming=True)
    for row in ds:
        instruction = row.get("instruction", "").strip()
        extra_input = row.get("input",       "").strip()
        response    = row.get("output",      "").strip()
        if not instruction or not response:
            continue
        prompt = f"{instruction}\n{extra_input}" if extra_input else instruction
        yield {"prompt": prompt, "response": response}


# ── Adapter 2: HuggingFaceH4/ultrachat_200k ──────────────────
def adapt_ultrachat_stream(split="train_sft"):
    """
    columns: messages (list of {role, content})
    prompt   = first user turn
    response = first assistant turn
    Note: uses train_sft split — ultrachat has no plain 'train' split
    """
    ds = load_dataset("HuggingFaceH4/ultrachat_200k",
                      split=split, streaming=True)
    for row in ds:
        messages = row.get("messages", [])
        prompt = response = None
        for msg in messages:
            role    = msg.get("role", "").lower()
            content = msg.get("content", "").strip()
            if not content:
                continue
            if role == "user" and prompt is None:
                prompt = content
            elif role == "assistant" and response is None:
                response = content
        if prompt and response:
            yield {"prompt": prompt, "response": response}


# ── Adapter 3: CohereLabs/aya_dataset ────────────────────────
def adapt_aya_stream(split="train"):
    """
    columns: inputs, targets
    prompt   = inputs
    response = targets
    """
    ds = load_dataset("CohereLabs/aya_dataset",
                      split=split, streaming=True)
    for row in ds:
        prompt   = str(row.get("inputs",  "")).strip()
        response = str(row.get("targets", "")).strip()
        if prompt and response:
            yield {"prompt": prompt, "response": response}


print("Adapters defined (generator-based, zero RAM buffering):")
print("  adapt_alpaca_stream    — tatsu-lab/alpaca")
print("  adapt_ultrachat_stream — HuggingFaceH4/ultrachat_200k")
print("  adapt_aya_stream       — CohereLabs/aya_dataset")

Adapters defined (generator-based, zero RAM buffering):
  adapt_alpaca_stream    — tatsu-lab/alpaca
  adapt_ultrachat_stream — HuggingFaceH4/ultrachat_200k
  adapt_aya_stream       — CohereLabs/aya_dataset


## 7. Stream Datasets Directly to JSONL

Each dataset streams from HuggingFace → adapter → disk.
One record in RAM at a time. No list buffering.

In [13]:
alpaca_jsonl = os.path.join(SFT_DIR, "alpaca.jsonl")

if not os.path.isfile(alpaca_jsonl):
    print("Streaming tatsu-lab/alpaca → alpaca.jsonl ...")
    n = stream_to_jsonl(adapt_alpaca_stream(), alpaca_jsonl, MAX_RECORDS)
    print(f"done: {n:,} records")
else:
    n = sum(1 for _ in open(alpaca_jsonl))
    print(f"alpaca.jsonl already exists: {n:,} records")

with open(alpaca_jsonl) as f:
    sample = json.loads(f.readline())
print(f"\nSample prompt   : {sample['prompt'][:80]}...")
print(f"Sample response : {sample['response'][:80]}...")

Streaming tatsu-lab/alpaca → alpaca.jsonl ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/7.47k [00:00<?, ?B/s]

  51,974 records written.          
done: 51,974 records

Sample prompt   : Give three tips for staying healthy....
Sample response : 1.Eat a balanced diet and make sure to include plenty of fruits and vegetables. ...


In [14]:
ultrachat_jsonl = os.path.join(SFT_DIR, "ultrachat.jsonl")

if not os.path.isfile(ultrachat_jsonl):
    print("Streaming HuggingFaceH4/ultrachat_200k (train_sft) → ultrachat.jsonl ...")
    n = stream_to_jsonl(adapt_ultrachat_stream(), ultrachat_jsonl, MAX_RECORDS)
    print(f"done: {n:,} records")
else:
    n = sum(1 for _ in open(ultrachat_jsonl))
    print(f"ultrachat.jsonl already exists: {n:,} records")

with open(ultrachat_jsonl) as f:
    sample = json.loads(f.readline())
print(f"\nSample prompt   : {sample['prompt'][:80]}...")
print(f"Sample response : {sample['response'][:80]}...")

Streaming HuggingFaceH4/ultrachat_200k (train_sft) → ultrachat.jsonl ...


README.md:   0%|          | 0.00/3.90k [00:00<?, ?B/s]

  207,865 records written.          
done: 207,865 records

Sample prompt   : These instructions apply to section-based themes (Responsive 6.0+, Retina 4.0+, ...
Sample response : This feature only applies to Collection pages and Featured Collections sections ...


In [15]:
aya_jsonl = os.path.join(SFT_DIR, "aya.jsonl")

if not os.path.isfile(aya_jsonl):
    print("Streaming CohereLabs/aya_dataset → aya.jsonl ...")
    n = stream_to_jsonl(adapt_aya_stream(), aya_jsonl, MAX_RECORDS)
    print(f"done: {n:,} records")
else:
    n = sum(1 for _ in open(aya_jsonl))
    print(f"aya.jsonl already exists: {n:,} records")

with open(aya_jsonl) as f:
    sample = json.loads(f.readline())
print(f"\nSample prompt   : {sample['prompt'][:80]}...")
print(f"Sample response : {sample['response'][:80]}...")

Streaming CohereLabs/aya_dataset → aya.jsonl ...


README.md:   0%|          | 0.00/13.7k [00:00<?, ?B/s]

  202,352 records written.          
done: 202,352 records

Sample prompt   : Heestan waxaa qada Khalid Haref Ahmed 
OO ku Jiray Kooxdii Dur Dur!...
Sample response : Habeen ma hurdoo
Aday horjoogoo
Dharaar ma hargalo
Aduun baabay helayee
Runtii k...


---
## 8. Tokenize — 3 JSONL Sources → Shards

`tokenize_dataset.py` launches one `SourceThread` per JSONL file,
merges them through `ShuffleBuffer`, tokenizes each record,
and writes shards to `out_sft_multi/`.

Binary format per shard (`samples.bin`):
```
uint32  n_samples
per sample:
    uint16  prompt_len
    uint32  prompt_ids[prompt_len]
    uint16  response_len
    uint32  response_ids[response_len]
```
No padding on disk. Padding built at batch time in C++ collator.

In [ ]:
out_dir = os.path.join(SFT_DIR, "out_sft_multi")
if os.path.isdir(out_dir):
    shutil.rmtree(out_dir)
    print(f"cleaned: {out_dir}")
else:
    print(f"output dir not found (ok): {out_dir}")

In [16]:
proc = subprocess.Popen(
    [
        "python3", "tokenize_dataset.py",
        "--input",               "alpaca.jsonl",
        "--input",               "ultrachat.jsonl",
        "--input",               "aya.jsonl",
        "--output-dir",          "out_sft_multi",
        "--tokenizer",           TOKENIZER,
        "--max-seq-len",         str(MAX_SEQ_LEN),
        "--shard-size",          str(SHARD_SIZE),
        "--shuffle-buffer-size", str(SHUFFLE_BUF),
    ],
    cwd=SFT_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

for line in proc.stdout:
    print(line, end="", flush=True)

proc.wait()
assert proc.returncode == 0, \
    f"tokenize_dataset.py failed with code {proc.returncode}"

[transformers] A new version of the following files was downloaded from https://huggingface.co/inceptionai/jais-family-590m:
- configuration_jais.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (3854 > 2048). Running this sequence through the model will result in indexing errors
[tokenize] loading tokenizer 'inceptionai/jais-family-590m' ...
[tokenize] tokenizer loaded  eos=0  pad=0
[tokenize] source: local JSONL → alpaca.jsonl
[tokenize] source: local JSONL → ultrachat.jsonl
[tokenize] source: local JSONL → aya.jsonl
[tokenize] starting 3 source thread(s) ...
  [shard] wrote 5000 samples → out_sft_multi/shard_00 (419 truncated)
[ShardFlusher] shard_00 written (5000 records)
  [shard] wrote 5000 samples → out_sft_multi/shard_01 (403 truncated)
[ShardFlusher] shard_01 writ

## 9. Verify Shards and Manifest

In [17]:
import struct
import os
out_dir = os.path.join(SFT_DIR, "out_sft_multi")   # ← add this

manifest_path = os.path.join(out_dir, "manifest.json")
assert os.path.isfile(manifest_path), "manifest.json NOT FOUND"

with open(manifest_path) as f:
    manifest = json.load(f)

print(f"manifest.json:")
print(f"  max_seq_len  : {manifest['max_seq_len']}")
print(f"  pad_token_id : {manifest['pad_token_id']}")
print(f"  total shards : {len(manifest['shards'])}")
print()

total_records = 0
for s in manifest["shards"]:
    shard_dir    = os.path.join(SFT_DIR, s["dir"])
    samples_path = os.path.join(shard_dir, "samples.bin")
    assert os.path.isfile(samples_path), f"MISSING: {samples_path}"
    size_mb = os.path.getsize(samples_path) / 1e6
    print(f"  shard_{s['shard']:02d}  "
          f"n_samples={s['n_samples']:>6,}  "
          f"samples.bin={size_mb:.2f} MB")
    total_records += s["n_samples"]

print(f"\ntotal records : {total_records:,}")
assert manifest["max_seq_len"] == MAX_SEQ_LEN
print("manifest verification: PASSED")

manifest.json:
  max_seq_len  : 512
  pad_token_id : 0
  total shards : 93

  shard_00  n_samples= 5,000  samples.bin=3.14 MB
  shard_01  n_samples= 5,000  samples.bin=2.98 MB
  shard_02  n_samples= 5,000  samples.bin=2.82 MB
  shard_03  n_samples= 5,000  samples.bin=2.84 MB
  shard_04  n_samples= 5,000  samples.bin=2.89 MB
  shard_05  n_samples= 5,000  samples.bin=3.15 MB
  shard_06  n_samples= 5,000  samples.bin=3.30 MB
  shard_07  n_samples= 5,000  samples.bin=3.32 MB
  shard_08  n_samples= 5,000  samples.bin=3.22 MB
  shard_09  n_samples= 5,000  samples.bin=3.76 MB
  shard_10  n_samples= 5,000  samples.bin=4.06 MB
  shard_11  n_samples= 5,000  samples.bin=4.37 MB
  shard_12  n_samples= 5,000  samples.bin=4.61 MB
  shard_13  n_samples= 5,000  samples.bin=4.76 MB
  shard_14  n_samples= 5,000  samples.bin=5.28 MB
  shard_15  n_samples= 5,000  samples.bin=5.61 MB
  shard_16  n_samples= 5,000  samples.bin=6.01 MB
  shard_17  n_samples= 5,000  samples.bin=6.30 MB
  shard_18  n_samples= 5

## 10. Inspect Binary Format

Read first 8 samples from `shard_00/samples.bin` directly in Python.
Verifies variable-length format — no pre-padding on disk.

In [18]:
def read_samples_preview(path, n_preview=8):
    results = []
    with open(path, "rb") as f:
        n_samples = struct.unpack("<I", f.read(4))[0]
        print(f"  n_samples in file = {n_samples:,}")
        for i in range(min(n_samples, n_preview)):
            prompt_len = struct.unpack("<H", f.read(2))[0]
            prompt_ids = struct.unpack(f"<{prompt_len}I",
                                        f.read(prompt_len * 4))
            resp_len   = struct.unpack("<H", f.read(2))[0]
            resp_ids   = struct.unpack(f"<{resp_len}I",
                                        f.read(resp_len * 4))
            results.append((prompt_len, resp_len))
            print(f"  sample {i:2d}: "
                  f"prompt_len={prompt_len:4d}  "
                  f"response_len={resp_len:4d}  "
                  f"total={prompt_len+resp_len:4d}  "
                  f"first_tok={prompt_ids[0]}  "
                  f"last_tok={resp_ids[-1]}")
    return results

shard0     = manifest["shards"][0]
shard0_path = os.path.join(SFT_DIR, shard0["dir"], "samples.bin")
print(f"Inspecting: {shard0_path}")
preview = read_samples_preview(shard0_path)

print()
for prompt_len, resp_len in preview:
    assert 0 < prompt_len  <= MAX_SEQ_LEN
    assert 0 < resp_len    <= MAX_SEQ_LEN

print("Binary format: PASSED")
print("  sequences stored at real length — no disk padding")

Inspecting: Optimized_data_loaders/sft_multiple_datasets/out_sft_multi/shard_00/samples.bin
  n_samples in file = 5,000
  sample  0: prompt_len=  39  response_len=  54  total=  93  first_tok=8773  last_tok=0
  sample  1: prompt_len=  19  response_len= 119  total= 138  first_tok=57  last_tok=0
  sample  2: prompt_len= 512  response_len= 512  total=1024  first_tok=35509  last_tok=0
  sample  3: prompt_len= 512  response_len= 512  total=1024  first_tok=35509  last_tok=0
  sample  4: prompt_len=  15  response_len=  39  total=  54  first_tok=21861  last_tok=0
  sample  5: prompt_len=  71  response_len= 421  total= 492  first_tok=22369  last_tok=0
  sample  6: prompt_len=  18  response_len=  94  total= 112  first_tok=12573  last_tok=0
  sample  7: prompt_len=  31  response_len=   7  total=  38  first_tok=64439  last_tok=0

Binary format: PASSED
  sequences stored at real length — no disk padding


## 11. Length Distribution

Reads all lengths from `shard_00` to show prompt vs response breakdown
and verify cross-source mixing via length variance.

In [19]:
import statistics

def read_all_lengths(path):
    prompt_lens = []
    resp_lens   = []
    with open(path, "rb") as f:
        n = struct.unpack("<I", f.read(4))[0]
        for _ in range(n):
            pl = struct.unpack("<H", f.read(2))[0]
            f.read(pl * 4)
            rl = struct.unpack("<H", f.read(2))[0]
            f.read(rl * 4)
            prompt_lens.append(pl)
            resp_lens.append(rl)
    return prompt_lens, resp_lens

print(f"Reading all lengths from shard_00 ...")
prompt_lens, resp_lens = read_all_lengths(shard0_path)
combined = [p + r for p, r in zip(prompt_lens, resp_lens)]

print(f"\n{'Metric':<15} {'Prompt':>10} {'Response':>10} {'Combined':>10}")
print("-" * 50)
for label, vals in [
    ("min",   [min(prompt_lens),  min(resp_lens),  min(combined)]),
    ("max",   [max(prompt_lens),  max(resp_lens),  max(combined)]),
    ("mean",  [sum(prompt_lens)/len(prompt_lens),
               sum(resp_lens)/len(resp_lens),
               sum(combined)/len(combined)]),
    ("stdev", [statistics.stdev(prompt_lens),
               statistics.stdev(resp_lens),
               statistics.stdev(combined)]),
]:
    print(f"  {label:<13} {vals[0]:>10.1f} {vals[1]:>10.1f} {vals[2]:>10.1f}")

print()
boundaries = [64, 128, 256, 512]
prev = 0
print("Combined length distribution:")
for b in boundaries:
    count = sum(1 for l in combined if prev < l <= b)
    pct   = 100 * count / len(combined)
    bar   = "█" * int(pct / 2)
    print(f"  ({prev:>3},{b:>4}] : {count:>5,}  ({pct:5.1f}%)  {bar}")
    prev = b
over = sum(1 for l in combined if l > boundaries[-1])
pct  = 100 * over / len(combined)
print(f"  (>{boundaries[-1]:>3}      : {over:>5,}  ({pct:5.1f}%)")

comb_std = statistics.stdev(combined)
print(f"\nCombined stdev = {comb_std:.1f}")
if comb_std > 30:
    print("High variance → 3 datasets mixed (ShuffleBuffer working)")
else:
    print("WARNING: low variance — sources may not be mixing")

Reading all lengths from shard_00 ...

Metric              Prompt   Response   Combined
--------------------------------------------------
  min                  4.0        2.0        8.0
  max                512.0      512.0     1024.0
  mean                48.6      107.3      155.8
  stdev               90.0      139.5      185.4

Combined length distribution:
  (  0,  64] : 2,009  ( 40.2%)  ████████████████████
  ( 64, 128] : 1,402  ( 28.0%)  ██████████████
  (128, 256] :   720  ( 14.4%)  ███████
  (256, 512] :   365  (  7.3%)  ███
  (>512      :   504  ( 10.1%)

Combined stdev = 185.4
High variance → 3 datasets mixed (ShuffleBuffer working)


---
## 12. Run C++ Binary

Passes `manifest.json` as `argv[1]`.

The C++ binary:
1. Parses `manifest.json` → discovers all shards
2. Reads `samples.bin` sequentially from each shard
3. `SequentialBatchSampler` builds shuffled batch index list
4. `CUDAPrefetcher` producer thread prepares batches concurrently
5. `TrimPaddingCollator` pads each batch to its own max real length
6. Transfers tensors to GPU non-blocking
7. Training loop consumes via `loader->next()`

In [20]:
manifest_rel = os.path.relpath(
    os.path.join(out_dir, "manifest.json"), SFT_DIR
)

print(f"Running: {os.path.basename(binary_path)} {manifest_rel}")
print("=" * 64)

run = subprocess.Popen(
    [os.path.abspath(binary_path), manifest_rel],
    cwd=SFT_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
for line in run.stdout:
    print(line, end="", flush=True)

run.wait()
assert run.returncode == 0, \
    f"binary exited with code {run.returncode}"

#Window 7 processed 4 shards = 20,000 records = 313 batches of 8 (with some remainder). The time breakdown is:

#disk load = 225 ms — reading 4 shards from Colab's filesystem in parallel. This is the I/O cost. 225 ms for 4 × 5,000 records means ~56 ms per shard which is reasonable for Colab's networked storage.
#intra-shard shuffle = 19.75 ms — shuffling 20,000 records in RAM. Pure CPU work, cheap.
#batch construction = 1887 ms — the dominant cost. 313 batches × ~6 ms average per batch = 1,878 ms. This is collation + H2D transfer for all 313 batches in this window.
#window total = 2133 ms — about 2.1 seconds per window of 4 shards.

Streaming output truncated to the last 5000 lines.
[producer] collate+transfer: 7.23 ms  batch_max_len=292
[producer] collate+transfer: 3.68 ms  batch_max_len=292
[producer] collate+transfer: 2.65 ms  batch_max_len=292
[producer] collate+transfer: 2.60 ms  batch_max_len=292
[producer] collate+transfer: 2.51 ms  batch_max_len=292
[producer] collate+transfer: 3.74 ms  batch_max_len=292
[producer] collate+transfer: 4.65 ms  batch_max_len=292
[producer] collate+transfer: 3.63 ms  batch_max_len=292
[producer] collate+transfer: 4.51 ms  batch_max_len=292
[producer] collate+transfer: 3.74 ms  batch_max_len=292
[producer] collate+transfer: 4.78 ms  batch_max_len=292
[producer] collate+transfer: 4.49 ms  batch_max_len=292
[producer] collate+transfer: 4.83 ms  batch_max_len=292
[producer] collate+transfer: 2.59 ms  batch_max_len=292
[producer] collate+transfer: 2.90 ms  batch_max_len=292
[producer] collate+transfer: 3.24 ms  batch_max_len=292
[producer] collate+transfer: 2.78 ms  batch_max_len=2

## 13. Final Summary

In [22]:
print("=" * 64)
print("PIPELINE SUMMARY")
print("=" * 64)
print()
print("Sources and field mapping:")
print("  tatsu-lab/alpaca             instruction+input → prompt")
print("                               output            → response")
print("  HuggingFaceH4/ultrachat_200k messages[user]    → prompt")
print("                               messages[asst]    → response")
print("  CohereLabs/aya_dataset       inputs            → prompt")
print("                               targets           → response")
print()
print("RAM strategy:")
print("  Stream → adapter → disk (one record at a time, no list buffering)")
print()
print(f"Tokenizer    : {TOKENIZER}")
print(f"max_seq_len  : {MAX_SEQ_LEN}")
print(f"shard_size   : {SHARD_SIZE:,}")
print(f"shuffle_buf  : {SHUFFLE_BUF:,}")
print()
print(f"Shards written : {len(manifest['shards'])}")
print(f"Total records  : {total_records:,}")
print()
print("What was tested:")
print("  Zero-RAM streaming adapters              : PASSED")
print("  3-source SourceThread parallel streaming : PASSED")
print("  ShuffleBuffer cross-source mixing        : PASSED")
print("  Variable-length binary format on disk    : PASSED")
print("  manifest.json written and parsed         : PASSED")
print("  C++ shard loading + batch construction   : PASSED")
print("  Per-batch trim padding collator          : PASSED")
print("  CUDAPrefetcher concurrent batch prep     : PASSED")

PIPELINE SUMMARY

Sources and field mapping:
  tatsu-lab/alpaca             instruction+input → prompt
                               output            → response
  HuggingFaceH4/ultrachat_200k messages[user]    → prompt
                               messages[asst]    → response
  CohereLabs/aya_dataset       inputs            → prompt
                               targets           → response

RAM strategy:
  Stream → adapter → disk (one record at a time, no list buffering)

Tokenizer    : inceptionai/jais-family-590m
max_seq_len  : 512
shard_size   : 5,000
shuffle_buf  : 20,000

Shards written : 93
Total records  : 462,191

What was tested:
  Zero-RAM streaming adapters              : PASSED
  3-source SourceThread parallel streaming : PASSED
  ShuffleBuffer cross-source mixing        : PASSED
  Variable-length binary format on disk    : PASSED
  manifest.json written and parsed         : PASSED
  C++ shard loading + batch construction   : PASSED
  Per-batch trim padding collator  